<a href="https://colab.research.google.com/github/yoshiagent/vibration-colab/blob/main/notebooks/Fourier_Spectrum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 傅立葉級數、頻譜與非弦波激振 Fourier Series, Spectrum and Nonsinusoidal Excitation

對照網頁：[振動學｜傅立葉分析與非弦波激振](https://yoshiagent.github.io/vibration/FO.html)

| 原檔 | 內容 | 本筆記本 |
|---|---|---|
| `FourierSq.m`、`FourierTriang.m` | 方波（50 項）、三角波（10 項）傅立葉級數部分和 | 第 1 節 |
| `GUI_FourierSeries2.dll` | 編譯過的 MATLAB GUI（無原始碼，無法改寫） | 第 1 節改用 Plotly 滑桿 |
| `freqspecsq.m`、`freqspec.m` | `periodogram` 頻譜：方波與兩個弦波 | 第 2 節 |
| `ft01.m`、`ft02.m` | 傅立葉轉換例題 1（指數）、例題 2（矩形脈波） | 第 3 節 |
| `nonsin.mdl`（Simulink） | 兩個弦波分別／合併通過同一個二階系統：疊加原理 | 第 4 節 |

只用 Colab 內建的 numpy、scipy、plotly。

In [1]:
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp, quad
import plotly.graph_objects as go


def show(fig, title, xlab="t (s)", ylab="f(t)", h=420):
    fig.update_layout(title=title, xaxis_title=xlab, yaxis_title=ylab, height=h, template="plotly_white",
                      margin=dict(l=60, r=20, t=50, b=50))
    fig.show()


square = lambda t: np.where(np.floor(t) % 2 == 0, 1.0, -1.0)          # f(t) = (-1)^k, k < t < k+1
triangle = lambda t: 1 - np.abs(((t + 1) % 2) - 1)                    # f(t) = 1 - |t|, |t| <= 1, 週期 2

## 1. 傅立葉級數部分和（`FourierSq.m`、`FourierTriang.m`）

```matlab
% FourierSq.m                              % FourierTriang.m
k=50; t=0:0.01:3; f=0;                     k=10; t=0:0.01:3; f=0.5;
for n = 1:k                                for n = 1:k
 bn=4/(2*n-1)/pi;                           an=4/(2*n-1)/(2*n-1)/pi/pi;
 f=f+bn*sin((2*n-1)*pi*t);                  f=f+an*cos((2*n-1)*pi*t);
end                                        end
```
方波：$f(t)=\sum\frac{4}{(2n-1)\pi}\sin(2n-1)\pi t$；三角波：$f(t)=\frac12+\sum\frac{4}{(2n-1)^2\pi^2}\cos(2n-1)\pi t$。
先用數值積分確認係數，再畫部分和。

In [2]:
for n in range(1, 6):
    bn = round(quad(lambda t: square(t) * np.sin(n * np.pi * t), 0, 2, points=[1])[0], 12) + 0.0   # + 0.0：去掉 -0.0
    an = round(quad(lambda t: triangle(t) * np.cos(n * np.pi * t), -1, 1, points=[0])[0], 12) + 0.0
    print(f"n = {n}: square b_n = {bn:+.6f} (formula {2 * (1 - (-1)**n) / (n * np.pi):+.6f}),  "
          f"triangle a_n = {an:+.6f} (formula {2 * (1 - (-1)**n) / (n * np.pi)**2:+.6f})")


def sq_sum(t, k):
    return sum(4 / (2 * n - 1) / np.pi * np.sin((2 * n - 1) * np.pi * t) for n in range(1, k + 1))


def tri_sum(t, k):
    return 0.5 + sum(4 / (2 * n - 1)**2 / np.pi**2 * np.cos((2 * n - 1) * np.pi * t) for n in range(1, k + 1))


t = np.arange(0, 3 + 1e-9, 0.01)
fig = go.Figure([go.Scatter(x=t, y=square(t), name="square wave", line=dict(color="#bbb")),
                 go.Scatter(x=t, y=sq_sum(t, 50), name="50 terms (FourierSq.m)")])
show(fig, "Square wave, 50-term Fourier series")
tf = np.linspace(0, 1, 200001)
print(f"Gibbs: max of 50-term square-wave sum = {sq_sum(tf, 50).max():.4f} (overshoot about 9% of the jump 2 -> 1.179)")
fig = go.Figure([go.Scatter(x=t, y=triangle(t), name="triangle wave", line=dict(color="#bbb")),
                 go.Scatter(x=t, y=tri_sum(t, 10), name="10 terms (FourierTriang.m)")])
show(fig, "Triangle wave, 10-term Fourier series")
print(f"triangle 10 terms: max |sum - f| = {np.max(np.abs(tri_sum(t, 10) - triangle(t))):.4f}  (no Gibbs: the triangle wave is continuous)")

n = 1: square b_n = +1.273240 (formula +1.273240),  triangle a_n = +0.405285 (formula +0.405285)
n = 2: square b_n = +0.000000 (formula +0.000000),  triangle a_n = +0.000000 (formula +0.000000)
n = 3: square b_n = +0.424413 (formula +0.424413),  triangle a_n = +0.045032 (formula +0.045032)
n = 4: square b_n = +0.000000 (formula +0.000000),  triangle a_n = +0.000000 (formula +0.000000)
n = 5: square b_n = +0.254648 (formula +0.254648),  triangle a_n = +0.016211 (formula +0.016211)


Gibbs: max of 50-term square-wave sum = 1.1790 (overshoot about 9% of the jump 2 -> 1.179)


triangle 10 terms: max |sum - f| = 0.0101  (no Gibbs: the triangle wave is continuous)


原課程的 `GUI_FourierSeries2.dll` 是可以調項數的 MATLAB GUI。這裡用 Plotly 滑桿代替：拖動滑桿改變項數（方波、三角波同時顯示）。

In [3]:
t = np.linspace(-1, 3, 801)
ks = [1, 2, 3, 5, 8, 12, 20, 35, 50]
fig = go.Figure([go.Scatter(x=t, y=square(t), line=dict(color="#ccc"), name="square"),
                 go.Scatter(x=t, y=triangle(t) + 3, line=dict(color="#ccc"), name="triangle (shifted +3)"),
                 go.Scatter(x=t, y=sq_sum(t, 1), name="square partial sum", line=dict(color="blue")),
                 go.Scatter(x=t, y=tri_sum(t, 1) + 3, name="triangle partial sum", line=dict(color="red"))])
fig.frames = [go.Frame(name=str(k), data=[go.Scatter(y=sq_sum(t, k)), go.Scatter(y=tri_sum(t, k) + 3)], traces=[2, 3]) for k in ks]
fig.update_layout(sliders=[dict(currentvalue=dict(prefix="number of terms k = "), steps=[
    dict(method="animate", label=str(k), args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True))]) for k in ks])])
show(fig, "Fourier partial sums (replaces GUI_FourierSeries2.dll)", h=520)

## 2. 頻譜（`freqspecsq.m`、`freqspec.m`）

```matlab
Fs=22; dt=1/Fs; nfft=Fs*10;            % 解析度 Fs/nfft = 0.1 Hz
t=0:dt:nfft*dt;
x(ii)=(-1)^floor(t(ii));               % freqspecsq.m：方波
x=2*sin(2*pi*5*t)+3*sin(2*pi*10.5*t);  % freqspec.m：兩個弦波
[Pxx,f]=periodogram(x,[],'onesided',nfft,Fs);
psdplot(sqrt(2*Pxx*Fs/length(x)),f,'Hz','linear','Spectrum')   % 換算成各頻率成分的振幅
stem(fn,bn)                            % 與傅立葉係數 bn=4/((2n-1)pi)、fn=(2n-1)/2 Hz 比較
```
單邊功率譜 $P_{xx}$ 換算成振幅：對 $A\sin(2\pi f_0t)$，峰值 $\sqrt{2P_{xx}F_s/N}=A$。
原程式取 `t=0:dt:nfft*dt` 共 nfft+1 點但 `nfft` 只有 220；這裡取前 220 點（剛好 10 秒、整數個週期）。
（`freqspec.m` 最後的 `specgram` 視窗長度等於整段訊號，只有一個時間切片，這裡不畫。）

In [4]:
Fs, nfft = 22, 220
t = np.arange(nfft) / Fs


def amp_spectrum(x):
    f, P = signal.periodogram(x, fs=Fs, nfft=nfft, window="boxcar", detrend=False, return_onesided=True, scaling="density")
    return f, np.sqrt(2 * P * Fs / len(x))


x = 2 * np.sin(2 * np.pi * 5 * t) + 3 * np.sin(2 * np.pi * 10.5 * t)
f, A = amp_spectrum(x)
fig = go.Figure(go.Scatter(x=f, y=A, name="amplitude"))
show(fig, "Spectrum of 2 sin(2 pi 5 t) + 3 sin(2 pi 10.5 t) (freqspec.m)", xlab="Frequency (Hz)", ylab="Magnitude")
for f0 in (5.0, 10.5):
    print(f"freqspec: peak at {f[np.argmin(np.abs(f - f0))]:.1f} Hz, amplitude = {A[np.argmin(np.abs(f - f0))]:.4f}")
print(f"Parseval: sum(x^2) = {np.sum(x**2):.2f}, (N/2) sum(A^2) = {len(x) / 2 * np.sum(A**2):.2f}  (MATLAB Pxxsum = sum(Pxx*Fs))")

xs = square(t + 1e-9)
f, A = amp_spectrum(xs)
fn = (2 * np.arange(1, 11) - 1) / 2
bn = 4 / (2 * np.arange(1, 11) - 1) / np.pi
fig = go.Figure([go.Scatter(x=f, y=A, name="periodogram amplitude"),
                 go.Scatter(x=fn, y=bn, mode="markers", marker=dict(size=9, color="red"), name="Fourier b_n (stem)")])
show(fig, "Square wave: spectrum vs Fourier coefficients (freqspecsq.m)", xlab="Frequency (Hz)", ylab="Magnitude")
for n in range(3):
    i = np.argmin(np.abs(f - fn[n]))
    print(f"freqspecsq: {fn[n]:.1f} Hz  spectrum = {A[i]:.4f}  b_n = {bn[n]:.4f}")
print("(small differences come from sampling: 22 samples per second of a square wave is not exactly band-limited)")

freqspec: peak at 5.0 Hz, amplitude = 2.0000
freqspec: peak at 10.5 Hz, amplitude = 3.0000
Parseval: sum(x^2) = 1430.00, (N/2) sum(A^2) = 1430.00  (MATLAB Pxxsum = sum(Pxx*Fs))


freqspecsq: 0.5 Hz  spectrum = 1.2743  b_n = 1.2732
freqspecsq: 1.5 Hz  spectrum = 0.4277  b_n = 0.4244
freqspecsq: 2.5 Hz  spectrum = 0.2601  b_n = 0.2546
(small differences come from sampling: 22 samples per second of a square wave is not exactly band-limited)


## 3. 傅立葉轉換例題（`ft01.m`、`ft02.m`）

```matlab
% ft01.m: h(t)=b*exp(-a*t), t>=0      % ft02.m: 矩形脈波，|t|<t0 時為 a
den=a*a+4*pi^2*f.^2;                  h=a/pi./f.*sin(2*pi*t0.*f)
realf=a*b./den;  imagf=-2*pi*b.*f./den
```
`ft02.m` 的 `a/pi./f` 在 $f=0$ 會出現 0/0（NaN，圖上缺一點）；極限值是 $2at_0$。這裡用 `np.sinc`：
$H(f)=2at_0\,\dfrac{\sin(2\pi t_0f)}{2\pi t_0f}=2at_0\,\mathrm{sinc}(2t_0f)$。兩題都用數值積分 $\int h(t)e^{-j2\pi ft}dt$ 驗證。

In [5]:
a = b = 1.0
f = np.arange(-2, 2 + 1e-9, 0.01)
den = a * a + 4 * np.pi**2 * f**2
H = (a * b - 2j * np.pi * b * f) / den
fig = go.Figure([go.Scatter(x=f, y=H.real, name="real", line=dict(color="red")), go.Scatter(x=f, y=H.imag, name="imaginary", line=dict(color="blue"))])
show(fig, "ft01: real(r) and imaginary(b) of Fourier transform", xlab="frequency (Hz)", ylab="H(f)")
fig = go.Figure([go.Scatter(x=f, y=np.abs(H), name="magnitude", line=dict(color="red")), go.Scatter(x=f, y=np.angle(H), name="phase (rad)", line=dict(color="blue"))])
show(fig, "ft01: magnitude(r) and phase(b) of Fourier transform", xlab="frequency (Hz)", ylab="")
err = max(abs(quad(lambda t: b * np.exp(-a * t) * np.cos(2 * np.pi * fk * t), 0, 50, limit=500)[0] - H[i].real)
          + abs(-quad(lambda t: b * np.exp(-a * t) * np.sin(2 * np.pi * fk * t), 0, 50, limit=500)[0] - H[i].imag)
          for i, fk in enumerate(f) if i % 40 == 0)
print(f"ft01: max |numerical FT - b/(a + j 2 pi f)| = {err:.1e}")

a, t0 = 1.0, 1.0
f = np.arange(-4, 4 + 1e-9, 0.01)
H2 = 2 * a * t0 * np.sinc(2 * t0 * f)
fig = go.Figure(go.Scatter(x=f, y=H2))
show(fig, "ft02: Fourier transform of a rectangular pulse", xlab="frequency (Hz)", ylab="H(f)")
err = max(abs(quad(lambda t: a * np.cos(2 * np.pi * fk * t), -t0, t0)[0] - H2[i]) for i, fk in enumerate(f) if i % 20 == 0)
print(f"ft02: max |numerical FT - 2 a t0 sinc(2 t0 f)| = {err:.1e}, H(0) = {H2[np.argmin(np.abs(f))]:.4f} (= 2 a t0)")

ft01: max |numerical FT - b/(a + j 2 pi f)| = 5.2e-10


ft02: max |numerical FT - 2 a t0 sinc(2 t0 f)| = 4.5e-16, H(0) = 2.0000 (= 2 a t0)


## 4. 非弦波激振與疊加原理（`nonsin.mdl`）

Simulink 模型的方塊（由 `.mdl` 檔讀出；StopTime = 10，ode45）：

* `Sine Wave`：振幅 1、頻率 1 rad/s → $f_1=\sin t$；`Sine Wave1`：振幅 2、頻率 2、相位 1 → $f_2=2\sin(2t+1)$
* 三個相同的 `Transfer Fcn`：$G(s)=\dfrac{25}{s^2+5s+25}$（$\omega_n=5$、$\zeta=0.5$）
* $y_1=G f_1$、$y_2=G f_2$、$y_3=G(f_1+f_2)$；`Sum1` $=y_1+y_2$，`Sum2` $=(y_1+y_2)-y_3$ 接到 `Scope4`

線性系統滿足疊加原理，所以 `Scope4` 應該是 0。這正是網頁上「把非弦波輸入展開成傅立葉級數、各項分別求響應再相加」的依據。

In [6]:
w_n, zeta = 5.0, 0.5
f1 = lambda t: np.sin(t)
f2 = lambda t: 2 * np.sin(2 * t + 1)


def sim(u, T=10.0):
    t = np.linspace(0, T, 2001)
    s = solve_ivp(lambda t, x: [x[1], 25 * u(t) - 5 * x[1] - 25 * x[0]], (0, T), [0, 0], t_eval=t, rtol=1e-10, atol=1e-12)
    return t, s.y[0]


t, y1 = sim(f1)
_, y2 = sim(f2)
_, y3 = sim(lambda t: f1(t) + f2(t))
fig = go.Figure([go.Scatter(x=t, y=f1(t) + f2(t), name="input f1 + f2", line=dict(color="#bbb")),
                 go.Scatter(x=t, y=y1, name="y1 = G f1 (Scope)"), go.Scatter(x=t, y=y2, name="y2 = G f2 (Scope1)"),
                 go.Scatter(x=t, y=y3, name="y3 = G (f1+f2) (Scope2)", line=dict(width=3)),
                 go.Scatter(x=t, y=y1 + y2 - y3, name="Sum2 = y1 + y2 - y3 (Scope4)", line=dict(color="black", dash="dot"))])
show(fig, "nonsin.mdl: superposition", ylab="")
print(f"nonsin.mdl: max |y1 + y2 - y3| = {np.max(np.abs(y1 + y2 - y3)):.1e}")
for w, A in ((1, 1), (2, 2)):
    G = 25 / (25 - w**2 + 5j * w)
    print(f"steady state of the w = {w} component: amplitude {A} x |G(j{w})| = {A * abs(G):.4f}, phase shift {np.degrees(np.angle(G)):.2f} deg")
t2, yl = sim(lambda t: f1(t) + f2(t), T=40)
tail = t2 > 30
ys = abs(25 / (24 + 5j)) * np.sin(t2 + np.angle(25 / (24 + 5j))) + 2 * abs(25 / (21 + 10j)) * np.sin(2 * t2 + 1 + np.angle(25 / (21 + 10j)))
print(f"steady state = sum of the two sinusoidal responses: max error after t = 30 s = {np.max(np.abs(yl[tail] - ys[tail])):.1e}")

nonsin.mdl: max |y1 + y2 - y3| = 9.1e-11
steady state of the w = 1 component: amplitude 1 x |G(j1)| = 1.0198, phase shift -11.77 deg
steady state of the w = 2 component: amplitude 2 x |G(j2)| = 2.1497, phase shift -25.46 deg


steady state = sum of the two sinusoidal responses: max error after t = 30 s = 8.2e-11
